# 03 Retention by Channel

В этом ноутбуке сравниваем retention и коммерческие метрики по `acquisition_channel`.

## 1. Load prepared transactions

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cohort import (
    calculate_channel_retention,
    calculate_country_retention,
    generate_product_hypotheses,
)
from src.visualization import (
    plot_retention_by_channel,
    plot_revenue_per_customer_by_channel,
    save_dataframe,
)

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
IMAGES_PATH = PROJECT_ROOT / "images"
PREPARED_FILE = PROCESSED_PATH / "transactions_prepared.csv"

date_columns = ["invoice_date", "order_month", "first_purchase_date", "cohort_month"]
transactions = pd.read_csv(PREPARED_FILE, parse_dates=date_columns)

print(f"Prepared shape: {transactions.shape}")
display(transactions.head())

## 2. Calculate retention by acquisition channel

Важно: `acquisition_channel` синтетический. Он добавлен для демонстрации сегментации, потому что в исходном Online Retail II dataset нет реального канала привлечения.

In [ ]:
channel_retention = calculate_channel_retention(transactions)

display(channel_retention.round(2))

## 3. Compare D7/D30/D90 retention

In [ ]:
plot_retention_by_channel(
    channel_retention,
    save_path=IMAGES_PATH / "d7_d30_d90_by_channel.png",
)
plt.show()

best_d30_channel = channel_retention.loc[channel_retention["d30_retention"].idxmax()]
best_d90_channel = channel_retention.loc[channel_retention["d90_retention"].idxmax()]

print(f"Best channel by D30 retention: {best_d30_channel['acquisition_channel']}")
print(f"Best channel by D90 retention: {best_d90_channel['acquisition_channel']}")

## 4. Revenue per customer by channel

In [ ]:
plot_revenue_per_customer_by_channel(
    channel_retention,
    save_path=IMAGES_PATH / "revenue_per_customer_by_channel.png",
)
plt.show()

best_revenue_channel = channel_retention.loc[channel_retention["revenue_per_customer"].idxmax()]
print(f"Best channel by revenue per customer: {best_revenue_channel['acquisition_channel']}")

display(
    channel_retention[
        ["acquisition_channel", "customers_count", "revenue", "revenue_per_customer"]
    ].sort_values("revenue_per_customer", ascending=False).round(2)
)

## 5. Average order value by channel

In [ ]:
display(
    channel_retention[
        ["acquisition_channel", "orders_count", "revenue", "average_order_value"]
    ].sort_values("average_order_value", ascending=False).round(2)
)

## 6. Country-level retention

In [ ]:
country_retention = calculate_country_retention(transactions, top_n=10)

display(country_retention.round(2))

## 7. Save results

In [ ]:
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

save_dataframe(channel_retention, PROCESSED_PATH / "channel_retention.csv")
save_dataframe(country_retention, PROCESSED_PATH / "country_retention.csv")

print("Saved channel and country retention outputs.")

## 8. Product hypotheses by segment

In [ ]:
hypotheses = generate_product_hypotheses(None, channel_retention)

for index, hypothesis in enumerate(hypotheses, start=1):
    print(f"{index}. {hypothesis}")

Интерпретация каналов требует осторожности: в этом проекте канал привлечения искусственный. Для реальной оценки нужны CAC, marketing spend, источник кампании и маржинальность.